Doctor speech
      ↓
Audio recording
      ↓
ASR model
      ↓
Transcript
      ↓
NLP processing
      ↓
Structured medical record

Doctor Audio
     ↓
Whisper ASR
     ↓
Transcript
     ↓
Medical NER
     ↓
Structured EMR

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

DATA_DIR_CANDIDATES = [
    Path.cwd() / "data" / "aci_bench_samples",
    Path.cwd().parent / "data" / "aci_bench_samples",
]

for candidate in DATA_DIR_CANDIDATES:
    if candidate.exists():
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not find data/aci_bench_samples")

GENERATED_NOTES_DIR = DATA_DIR / "generated_notes"
RESULTS_DIR = Path.cwd() / "evaluation"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Using data directory: {DATA_DIR}")
print(f"Using generated-notes directory: {GENERATED_NOTES_DIR}")

#example
Audio:
"The patient has been coughing for two weeks."

ASR output:
"The patient has been coughing for two weeks."

NLP extraction:
Symptom: cough
Duration: 2 weeks

In [2]:
from datasets import load_dataset

# Load specific subset and split
dataset = load_dataset('ekacare/eka-medical-asr-evaluation-dataset', 'en', split='test')

# Load all splits from a subset
dataset = load_dataset('ekacare/eka-medical-asr-evaluation-dataset', 'en')

# Load everything
dataset = load_dataset('ekacare/eka-medical-asr-evaluation-dataset')

print (dataset)


Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for 

DatasetDict({
    test: Dataset({
        features: ['md5_text', 'file_name', 'audio', 'md5_audio', 'duration', 'text', 'audio_language', 'text_language', 'session_id', 'speaker', 'type_concept', 'recording_context', 'medical_entities'],
        num_rows: 3619
    })
})


In [3]:
import os
import pandas as pd
from datasets import load_dataset

# Load dataset and remove audio column to avoid torch/torchcodec dependency
dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", trust_remote_code=True)
dataset = dataset.remove_columns(['audio'])

# Get only the text data
rows = []
for i, sample in enumerate(dataset["test"]):
    filename = f"audio_{i}.wav"
    text = sample["text"]
    
    rows.append({
        "audio_path": f"audio/{filename}",
        "transcript": text
    })

# Convert to table
df = pd.DataFrame(rows)

# Create dataset folder
os.makedirs("medical_asr_dataset", exist_ok=True)

# Save metadata file
df.to_csv("medical_asr_dataset/metadata.csv", index=False)

print("Dataset table created!")
print("Total samples:", len(df))
print(df.head())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ekacare/eka-medical-asr-evaluation-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Dataset table created!
Total samples: 3619
          audio_path                                         transcript
0  audio/audio_0.wav  not having adequate rest. Okay okay. So that c...
1  audio/audio_1.wav  2 times in a day, please have, an antibiotic n...
2  audio/audio_2.wav  500 mg. Also, Because you're feeling weak. Tak...
3  audio/audio_3.wav  Patient has fever, headache, back pain, leg pa...
4  audio/audio_4.wav  Gelusil tablet and many more drugs and see aft...


In [4]:
pip install soundfile

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import csv
import soundfile as sf
from datasets import load_dataset

# Load dataset WITHOUT loading audio to avoid torchcodec dependency
dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", trust_remote_code=True)

# Check if audio column exists and try to load audio on-demand
os.makedirs("audio_files", exist_ok=True)

try:
    # This approach removes audio column to avoid torchcodec issues
    dataset = dataset.remove_columns(['audio'])
    print("Audio column removed. Transcripts only will be saved.")
    
    with open("metadata.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["file", "text"])
        
        for i, sample in enumerate(dataset["test"]):
            text = sample["text"]
            filename = f"audio_{i}.wav"
            writer.writerow([filename, text])
    
    print("Done! Transcripts saved in metadata.csv")
    print(f"Note: Audio files were not saved due to torchcodec dependency. Transcripts are available.")
    
except Exception as e:
    print(f"Error: {e}")
    print("Alternative: Use the dataset without audio column (see earlier cells)")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ekacare/eka-medical-asr-evaluation-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Audio column removed. Transcripts only will be saved.
Done! Transcripts saved in metadata.csv
Note: Audio files were not saved due to torchcodec dependency. Transcripts are available.


In [6]:
import os
import pandas as pd
from datasets import load_dataset

# Load dataset and remove audio column to avoid torch/torchcodec dependency
dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", trust_remote_code=True)
dataset = dataset.remove_columns(['audio'])

rows = []

for i, sample in enumerate(dataset["test"]):
    filename = f"audio_{i}.wav"
    text = sample["text"]

    rows.append({
        "audio_path": f"audio/{filename}",
        "transcript": text
    })

df = pd.DataFrame(rows)

# create folder if it doesn't exist
os.makedirs("medical_asr_dataset", exist_ok=True)

df.to_csv("medical_asr_dataset/metadata.csv", index=False)

print("Dataset table created!")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ekacare/eka-medical-asr-evaluation-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Dataset table created!


In [7]:
df.head(20)

,audio_path,transcript
0,audio/audio_0.wav,not having adequate rest. Okay okay. So that c...
1,audio/audio_1.wav,"2 times in a day, please have, an antibiotic n..."
2,audio/audio_2.wav,"500 mg. Also, Because you're feeling weak. Tak..."
3,audio/audio_3.wav,"Patient has fever, headache, back pain, leg pa..."
4,audio/audio_4.wav,Gelusil tablet and many more drugs and see aft...
5,audio/audio_5.wav,"Hello. The patient has fever, headache, body a..."
6,audio/audio_6.wav,"And, I want to also give Pantop DSR 40. And th..."
7,audio/audio_7.wav,"Patient has headache, fever, depression, leg p..."
8,audio/audio_8.wav,"For the medicine, take thyroxine. Also take Do..."
9,audio/audio_9.wav,"plus,there was this issue of,stomach ache, and..."


In [ ]:
# ACI-Bench vs generated note evaluation
# Drop your pipeline outputs into evaluation/generated_notes/<sample_name>.txt
# or pass a generator function that returns the note text for each transcript.

from pathlib import Path
import json

import pandas as pd
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def compute_semantic_similarity(source_text, target_text):
    if not source_text or not target_text:
        return 0.0
    
    vectorizer = TfidfVectorizer()
    try:
        # Convert the two notes into frequency vectors
        tfidf = vectorizer.fit_transform([source_text, target_text])
        # Calculate the angle between the vectors
        sim = cosine_similarity(tfidf[0:1], tfidf[1:2])
        return float(sim[0][0])
    except ValueError:
        # Handle cases with no overlapping words or empty strings
        return 0.0
    
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'aci_bench_samples').exists():
            return candidate
    return Path.cwd()


REPO_ROOT = resolve_repo_root()
ACI_BENCH_DIR = REPO_ROOT / 'data' / 'aci_bench_samples'
GENERATED_NOTES_DIR = REPO_ROOT / 'evaluation' / 'generated_notes'
OUTPUT_CSV = REPO_ROOT / 'evaluation' / 'aci_bench_note_comparison.csv'
OUTPUT_JSON = REPO_ROOT / 'evaluation' / 'aci_bench_note_comparison_summary.json'

SECTION_ALIASES = {
    'chief_complaint': ['CHIEF COMPLAINT'],
    'hpi': ['HISTORY OF PRESENT ILLNESS', 'HPI'],
    'ros': ['REVIEW OF SYSTEMS'],
    'physical_exam': ['PHYSICAL EXAMINATION', 'OBJECTIVE'],
    'results': ['RESULTS'],
    'assessment': ['ASSESSMENT'],
    'plan': ['PLAN', 'ASSESSMENT AND PLAN'],
}
SECTION_GROUPS = {
    'chief_complaint': [
        'CHIEF COMPLAINT', 'CC:', 'CC ', 'PRESENTING COMPLAINT', 'CHIEF COMPLAINT:'
    ],
    'hpi': [
        'HISTORY OF PRESENT ILLNESS', 'HPI', 'HPI:', 'HISTORY OF THE PRESENT ILLNESS',
        'PRESENT ILLNESS', 'HISTORY OF PRESENTING'
    ],
    'ros': [
        'REVIEW OF SYSTEMS', 'ROS', 'SYSTEMS REVIEW', 'ROS:'
    ],
    'physical_exam': [
        'PHYSICAL EXAMINATION', 'PHYSICAL EXAM', 'OBJECTIVE', 'EXAM:', 'EXAM\n',
        'PE:', 'EXAMINATION', 'PHYSICAL FINDINGS'
    ],
    'results': [
        'RESULTS', 'RESULTS:', 'DIAGNOSTIC RESULTS', 'LABORATORY',
        'LAB RESULTS', 'IMAGING', 'TEST RESULTS', 'DIAGNOSTICS'
    ],
    'assessment_plan': [
        'ASSESSMENT AND PLAN', 'ASSESSMENT & PLAN', 'ASSESSMENT/PLAN',
        'ASSESSMENT', 'PLAN', 'PLAN:', 'IMPRESSION AND PLAN',
        'IMPRESSION', 'IMPRESSION:'
    ],
    'medications': [
        'MEDICATIONS', 'CURRENT MEDICATIONS', 'MEDICATION LIST',
        'CURRENT MEDICATIONS:', 'MEDICATIONS:'
    ],
    'medical_history': [  # NEW
        'PAST MEDICAL HISTORY', 'MEDICAL HISTORY', 'PMH', 'PMH:',
        'PAST MEDICAL HISTORY:', 'RELEVANT MEDICAL HISTORY'
    ],
    'surgical_history': [  # NEW
        'PAST SURGICAL HISTORY', 'SURGICAL HISTORY', 'PSH', 'PSH:', 'PAST SURGERY',
        'PAST SURGICAL HISTORY:'
    ],
}



def read_text(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return handle.read().strip()


def load_aci_pairs(sample_dir=ACI_BENCH_DIR):
    pairs = []
    for transcript_path in sorted(sample_dir.glob('*_transcript.txt')):
        sample_name = transcript_path.name.replace('_transcript.txt', '')
        groundtruth_path = sample_dir / f'{sample_name}_groundtruth.txt'
        if groundtruth_path.exists():
            pairs.append({
                'sample_name': sample_name,
                'transcript': read_text(transcript_path),
                'groundtruth': read_text(groundtruth_path),
            })
    if not pairs:
        raise RuntimeError(f'No ACI-Bench transcript/groundtruth pairs found in {sample_dir}')
    return pairs


def load_generated_note(sample_name, transcript, generator_fn=None):
    if callable(generator_fn):
        return (generator_fn(transcript, sample_name=sample_name) or '').strip()

    for candidate in [
        GENERATED_NOTES_DIR / f'{sample_name}.txt',
        GENERATED_NOTES_DIR / f'{sample_name}_generated.txt',
        GENERATED_NOTES_DIR / f'{sample_name}_note.txt',
    ]:
        if candidate.exists():
            return read_text(candidate)
    return ''
from jiwer import wer, transforms
import jiwer

# With normalization pipeline — important for clinical text
TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.ExpandCommonEnglishContractions(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(), # <--- ADD THIS LINE
])

def compute_wer(reference: str, hypothesis: str) -> float:
    return wer(reference, hypothesis, 
               reference_transform=TRANSFORM,
               hypothesis_transform=TRANSFORM)

def extract_section_text(text: str, section_group: str) -> str:
    """Extract text belonging to a specific section."""
    clean = re.sub(r'\*+', '', text)
    upper = clean.upper()
    
    aliases = SECTION_GROUPS.get(section_group, [])
    
    # Find where this section starts
    start_idx = -1
    for alias in aliases:
        idx = upper.find(alias)
        if idx != -1:
            if start_idx == -1 or idx < start_idx:
                start_idx = idx
    
    if start_idx == -1:
        return ''
    
    # Find where the next section starts
    all_headers = [
        alias 
        for group in SECTION_GROUPS.values() 
        for alias in group
    ]
    
    end_idx = len(text)
    for header in all_headers:
        idx = upper.find(header, start_idx + 1)
        if idx != -1 and idx < end_idx:
            end_idx = idx
    
    return clean[start_idx:end_idx].strip()


def compute_section_wer(reference_text: str, generated_text: str) -> dict:
    """WER computed per section, only for sections present in both."""
    ref_sections = detect_sections(reference_text)
    gen_sections = detect_sections(generated_text)
    shared = ref_sections & gen_sections
    
    section_wers = {}
    for section in shared:
        ref_text = extract_section_text(reference_text, section)
        gen_text = extract_section_text(generated_text, section)
        if ref_text and gen_text:
            section_wers[section] = compute_wer(ref_text, gen_text)
    
    if section_wers:
        section_wers['mean'] = sum(section_wers.values()) / len(section_wers)
    
    return section_wers

def detect_sections(text):
    upper = (text or '').upper()
    sections = set()
    for section_name, aliases in SECTION_ALIASES.items():
        if any(alias in upper for alias in aliases):
            sections.add(section_name)
    return sections

def normalize_for_rouge(text: str) -> str:
    """Remove markdown formatting that doesn't exist in ground truth."""
    text = re.sub(r'\*+', '', text)           # remove ** bold and * italic
    text = re.sub(r'#+\s*', '', text)          # remove ## headers
    text = re.sub(r'-\s+(?=[A-Z])', '', text)  # remove bullet points before capitalized text
    text = re.sub(r'\[.*?\]', '', text)        # remove [UNCERTAIN - ...] placeholders
    text = re.sub(r'\s+', ' ', text).strip()   # collapse whitespace
    return text

# Sections that only count if the reference also has them
OPTIONAL_SECTIONS = {'results', 'medications', 'ros'}

def detect_sections(text: str) -> set:
    """
    Returns a set of canonical section group names found in text.
    Matches header-like lines (ALL CAPS or bold markdown) loosely.
    """
    if not text:
        return set()

    upper = text.upper()
    found = set()

    for group_name, aliases in SECTION_GROUPS.items():
        for alias in aliases:
            # Match as a header: alias appears after newline or at start,
            # optionally followed by colon, bold markers, or newline
            if alias in upper:
                found.add(group_name)
                break

    return found


def has_section_content(text: str, group_name: str) -> bool:
    """
    Fallback: check if section content appears even without a proper header.
    Uses content signals rather than header keywords.
    """
    upper = text.upper()
    content_signals = {
        'chief_complaint': ['CHIEF COMPLAINT', 'PRESENTING WITH', 'COMES IN FOR', 'HERE FOR'],
        'hpi': ['REPORTS ', 'PRESENTS WITH', 'HISTORY OF', 'PATIENT STATES', 'PATIENT REPORTS'],
        'ros': ['DENIES ', 'ENDORSES ', 'REVIEW OF'],
        'physical_exam': ['BLOOD PRESSURE', 'HEART RATE', 'BILATERAL', 'EDEMA', 'MURMUR', 'AUSCULTATION'],
        'results': ['ECHOCARDIOGRAM', 'X-RAY', 'EKG', 'EJECTION FRACTION', 'LAB', 'MG/DL'],
        'assessment_plan': ['INCREASE ', 'CONTINUE ', 'ORDER ', 'FOLLOW-UP', 'LISINOPRIL', 'LASIX'],
    }
    return any(signal in upper for signal in content_signals.get(group_name, []))


def compute_section_coverage(reference_text: str, generated_text: str,
                              partial_credit: float = 0.5) -> dict:
    ref_sections = detect_sections(reference_text)
    gen_sections = detect_sections(generated_text)
    required = ref_sections.copy()

    if not required:
        return {'section_coverage': 1.0, 'reference_sections': set(),
                'generated_sections': gen_sections, 'matched_sections': set(),
                'missing_sections': set()}

    score = 0.0
    matched, partial, missing = set(), set(), set()

    for section in required:
        if section in gen_sections:
            score += 1.0
            matched.add(section)
        elif has_section_content(generated_text, section):
            score += partial_credit   # content present, header missing
            partial.add(section)
        else:
            missing.add(section)

    return {
        'section_coverage': score / len(required),
        'reference_sections': required,
        'generated_sections': gen_sections,
        'matched_sections': matched,
        'partial_sections': partial,
        'missing_sections': missing,
    }

def evaluate_aci_bench(generator_fn=None):
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rows = []

    for sample in load_aci_pairs():
        generated = load_generated_note(sample['sample_name'], sample['transcript'], generator_fn=generator_fn)
        if not generated:
            print(f"[SKIP] No generated note found for {sample['sample_name']}")
            continue

        scores = rouge.score(
            normalize_for_rouge(sample['groundtruth']),
            normalize_for_rouge(generated)
        )
        coverage_result = compute_section_coverage(sample['groundtruth'], generated)
        section_coverage = coverage_result['section_coverage']
        reference_sections = coverage_result['reference_sections']
        generated_sections = coverage_result['generated_sections']

        # Optional: log missing sections for debugging
        if coverage_result['missing_sections']:
            print(f"[{sample['sample_name']}] Missing sections: {coverage_result['missing_sections']}")

        semantic_sim = compute_semantic_similarity(sample['groundtruth'], generated)
        rows.append({
            'sample_name': sample['sample_name'],
            'rouge1': scores['rouge1'].fmeasure,
            'rouge2': scores['rouge2'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure,
            'section_coverage': section_coverage,
            'reference_section_count': len(reference_sections),
            'generated_section_count': len(generated_sections),
            'wer_overall': min(compute_wer(sample['groundtruth'], generated), 1.0),
            'semantic_similarity': semantic_sim,
        })

    if not rows:
        raise RuntimeError(
            f'No generated notes were found. Put notes in {GENERATED_NOTES_DIR} '
            'or pass a generator_fn that returns a note for each transcript.'
        )

    results_df = pd.DataFrame(rows)
    summary = {
        'samples_evaluated': int(len(results_df)),
        'rouge1_mean': float(results_df['rouge1'].mean()),
        'rouge2_mean': float(results_df['rouge2'].mean()),
        'rougeL_mean': float(results_df['rougeL'].mean()),
        'section_coverage_mean': float(results_df['section_coverage'].mean()),
    }

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUTPUT_CSV, index=False)
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=4)

    print('Evaluation complete.')
    print(f'Saved per-sample results to: {OUTPUT_CSV}')
    print(f'Saved summary to: {OUTPUT_JSON}')
    print(summary)

    return results_df, summary


# Example:
# results_df, summary = evaluate_aci_bench(generator_fn=your_pipeline_function)

if any(GENERATED_NOTES_DIR.glob('*.txt')):
    results_df, summary = evaluate_aci_bench()
    print(results_df.head())
else:
    print(f'Place generated notes in {GENERATED_NOTES_DIR} or pass a generator_fn, then run evaluate_aci_bench().')

In [9]:
from pathlib import Path
import sys
import json

def resolve_repo_root_for_pipeline() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'pipeline' / 'medical_pipeline.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root with pipeline/medical_pipeline.py')


REPO_ROOT = resolve_repo_root_for_pipeline()
PIPELINE_DIR = REPO_ROOT / 'pipeline'
ACI_BENCH_DIR = REPO_ROOT / 'data' / 'aci_bench_samples'
GENERATED_NOTES_DIR = REPO_ROOT / 'evaluation' / 'generated_notes'
API_TOKEN_PATH = REPO_ROOT / '.api_token.json'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from medical_pipeline import (
    summarize_transcript,
    load_existing_vectorstore,
    get_relevant_context,
    suggest_icd_codes_local,
    CHROMA_PATH,
    ICD_CODES_PATH,
)


def read_text(path: Path) -> str:
    with open(path, 'r', encoding='utf-8') as handle:
        return handle.read().strip()


def load_groq_token() -> str:
    if not API_TOKEN_PATH.exists():
        raise FileNotFoundError(f'Missing API token file: {API_TOKEN_PATH}')
    with open(API_TOKEN_PATH, 'r', encoding='utf-8') as handle:
        tokens = json.load(handle)
    groq_token = tokens.get('groq-token')
    if not groq_token or groq_token == 'your-groq-api-token':
        raise RuntimeError('Groq token is missing or placeholder in .api_token.json')
    return groq_token


def generate_note_for_sample(sample_name: str, use_rag: bool = False) -> Path:
    transcript_path = ACI_BENCH_DIR / f'{sample_name}_transcript.txt'
    if not transcript_path.exists():
        raise FileNotFoundError(f'Transcript not found: {transcript_path}')

    transcript = read_text(transcript_path)
    groq_token = load_groq_token()

    rag_context = ''
    suggested_codes = []
    if callable(suggest_icd_codes_local):
        suggested_codes = suggest_icd_codes_local(transcript, icd_path=ICD_CODES_PATH, top_k=3)

    if use_rag and callable(load_existing_vectorstore) and callable(get_relevant_context):
        vectorstore = load_existing_vectorstore(chroma_path=CHROMA_PATH)
        if vectorstore is not None:
            rag_context = get_relevant_context(
                transcript,
                vectorstore,
                final_k=5,
                retrieve_k=12,
                max_queries=8,
            )

    note_text = summarize_transcript(
        transcript,
        groq_token,
        rag_context=rag_context,
        suggested_codes=suggested_codes,
    )

    GENERATED_NOTES_DIR.mkdir(parents=True, exist_ok=True)
    output_path = GENERATED_NOTES_DIR / f'{sample_name}.txt'
    with open(output_path, 'w', encoding='utf-8') as handle:
        handle.write(note_text)

    print(f'Generated note: {output_path}')
    return output_path


def generate_notes_for_samples(sample_names=None, use_rag: bool = False):
    if sample_names is None:
        sample_names = [
            p.name.replace('_transcript.txt', '')
            for p in sorted(ACI_BENCH_DIR.glob('*_transcript.txt'))
        ]

    output_files = []
    for sample_name in sample_names:
        output_files.append(generate_note_for_sample(sample_name, use_rag=use_rag))

    print(f'Generated {len(output_files)} note(s) in {GENERATED_NOTES_DIR}')
    return output_files


In [ ]:
# Full run: generate notes for all ACI transcripts, then evaluate vs ground truth
#generate_notes_for_samples(use_rag=False)
results_df, summary = evaluate_aci_bench()
print(summary)
results_df

Evaluation: ROUGE is weak metric for this task. From closer examination of ground truth vs generated sample, the generated sample includes more details from the transcript, which is penalized by ROUGE.
WER is also high due to extremely simplified ground truth notes.

## Transcription-Only Evaluation (EKA Clips + Gladia)
This section evaluates transcription quality by:
1. Loading local EKA clips from `data/eka_dataset_audio`
2. Using ground-truth transcripts from `data/eka_dataset_transcripts/metadata.csv`
3. Transcribing each available clip with `transcribe_audio(...)` from `pipeline/medical_pipeline.py`
4. Computing WER, CER, MER, WIL, WIP, ROUGE, and semantic similarity
5. Saving per-sample and aggregate results under `evaluation/`

In [ ]:
# Transcription-only evaluation on EKA dataset clips using Gladia ASR
from pathlib import Path
import json
import re
import string
import sys

import pandas as pd
from jiwer import cer, mer, wer, wil, wip
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _resolve_repo_root_for_pipeline() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "pipeline" / "medical_pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root with pipeline/medical_pipeline.py")


REPO_ROOT = _resolve_repo_root_for_pipeline()
PIPELINE_DIR = REPO_ROOT / "pipeline"
API_TOKEN_PATH = REPO_ROOT / ".api_token.json"

# Reuse medical pipeline transcription function
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from medical_pipeline import transcribe_audio


def _load_gladia_token(token_path: Path) -> str:
    if not token_path.exists():
        raise FileNotFoundError(f"Missing token file: {token_path}")
    with open(token_path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)
    token = payload.get("gladia-token")
    if not token or token == "your-gladia-api-token":
        raise RuntimeError("Gladia token missing or placeholder in .api_token.json")
    return token


def _strip_speaker_tags(text: str) -> str:
    return re.sub(r"\bSPEAKER_\d+\s*:\s*", "", text or "")


def _normalize_text(text: str) -> str:
    text = _strip_speaker_tags(text)
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _semantic_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    vec = TfidfVectorizer()
    try:
        tfidf = vec.fit_transform([a, b])
        return float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0])
    except ValueError:
        return 0.0


def _resolve_eka_paths(repo_root: Path):
    metadata_path = repo_root / "data" / "eka_dataset_transcripts" / "metadata.csv"
    audio_dirs = [
        repo_root / "data" / "eka_dataset_audio" / "audio sample",
        repo_root / "data" / "eka_dataset_audio" / "audio",
        repo_root / "data" / "eka_dataset_audio",
    ]
    return metadata_path, audio_dirs


def _find_audio_from_metadata(audio_path_value: str, audio_dirs: list[Path]) -> Path | None:
    rel_path = Path(str(audio_path_value).strip())
    candidates = [rel_path.name, str(rel_path).replace("\\", "/").split("/")[-1]]
    candidates = [c for c in candidates if c]

    for base_dir in audio_dirs:
        if not base_dir.exists():
            continue

        # Try exact relative path first
        exact = base_dir / rel_path
        if exact.exists() and exact.is_file():
            return exact

        # Try by filename
        for name in candidates:
            f = base_dir / name
            if f.exists() and f.is_file():
                return f

    return None


def evaluate_eka_transcription(
    max_samples: int | None = None,
    force_retranscribe: bool = False,
    save_predictions: bool = True,
):
    metadata_path, audio_dirs = _resolve_eka_paths(REPO_ROOT)
    if not metadata_path.exists():
        raise FileNotFoundError(f"EKA metadata not found: {metadata_path}")

    df_meta = pd.read_csv(metadata_path)
    required_cols = {"audio_path", "transcript"}
    if not required_cols.issubset(set(df_meta.columns)):
        raise RuntimeError(
            f"Metadata must contain columns {required_cols}, found: {list(df_meta.columns)}"
        )

    if max_samples is not None:
        df_meta = df_meta.head(max_samples).copy()

    pred_out_dir = REPO_ROOT / "evaluation" / "generated_transcriptions" / "eka"
    metrics_csv_path = REPO_ROOT / "evaluation" / "eka_gladia_transcription_metrics.csv"
    metrics_json_path = REPO_ROOT / "evaluation" / "eka_gladia_transcription_summary.json"
    pred_out_dir.mkdir(parents=True, exist_ok=True)

    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rows = []
    missing_audio_rows = []

    # Map metadata rows to available local clips
    row_audio_map = {}
    for i, row in df_meta.iterrows():
        audio_ref = str(row["audio_path"]).strip()
        audio_file = _find_audio_from_metadata(audio_ref, audio_dirs)
        if audio_file is None:
            missing_audio_rows.append(int(i))
        else:
            row_audio_map[int(i)] = audio_file

    if not row_audio_map:
        summary = {
            "samples_requested": int(len(df_meta)),
            "samples_with_audio": 0,
            "samples_evaluated": 0,
            "missing_audio_rows": missing_audio_rows,
            "note": "No local EKA clips found for metadata rows. Put clips under data/eka_dataset_audio/audio sample.",
        }
        with open(metrics_json_path, "w", encoding="utf-8") as handle:
            json.dump(summary, handle, indent=2)
        print(json.dumps(summary, indent=2))
        return pd.DataFrame(), summary

    gladia_token = _load_gladia_token(API_TOKEN_PATH)

    for i, row in df_meta.iterrows():
        row_id = int(i)
        audio_file = row_audio_map.get(row_id)
        if audio_file is None:
            continue

        reference_raw = str(row["transcript"] or "").strip()
        if not reference_raw:
            continue

        stem = Path(str(row["audio_path"])).stem or f"row_{row_id}"
        pred_path = pred_out_dir / f"{stem}_gladia.txt"
        sentence_conf_count = 0
        avg_conf = None

        if pred_path.exists() and not force_retranscribe:
            predicted_raw = pred_path.read_text(encoding="utf-8").strip()
            print(f"[row {row_id}] Using cached transcript: {pred_path.name}")
        else:
            print(f"[row {row_id}] Transcribing {audio_file.name} with Gladia...")
            try:
                predicted_raw, sentence_confidences, avg_conf = transcribe_audio(
                    str(audio_file), gladia_token=gladia_token
                )
                sentence_conf_count = len(sentence_confidences or [])
            except SystemExit as ex:
                print(f"[row {row_id}] Gladia failed (SystemExit={ex}). Skipping.")
                continue

            if save_predictions:
                pred_path.write_text(predicted_raw, encoding="utf-8")

        ref_norm = _normalize_text(reference_raw)
        pred_norm = _normalize_text(predicted_raw)
        if not ref_norm or not pred_norm:
            print(f"[row {row_id}] Empty normalized text, skipping.")
            continue

        rouge_scores = rouge.score(ref_norm, pred_norm)

        rows.append(
            {
                "row_id": row_id,
                "audio_file": str(audio_file.relative_to(REPO_ROOT)).replace("\\", "/"),
                "metadata_audio_path": str(row["audio_path"]),
                "reference_chars": len(reference_raw),
                "prediction_chars": len(predicted_raw),
                "wer": min(wer(ref_norm, pred_norm), 1.0),
                "cer": min(cer(ref_norm, pred_norm), 1.0),
                "mer": min(mer(ref_norm, pred_norm), 1.0),
                "wil": min(wil(ref_norm, pred_norm), 1.0),
                "wip": max(0.0, min(wip(ref_norm, pred_norm), 1.0)),
                "rouge1": rouge_scores["rouge1"].fmeasure,
                "rouge2": rouge_scores["rouge2"].fmeasure,
                "rougeL": rouge_scores["rougeL"].fmeasure,
                "semantic_similarity": _semantic_similarity(ref_norm, pred_norm),
                "gladia_avg_sentence_confidence": avg_conf,
                "gladia_sentence_segments": sentence_conf_count,
            }
        )

    if not rows:
        summary = {
            "samples_requested": int(len(df_meta)),
            "samples_with_audio": int(len(row_audio_map)),
            "samples_evaluated": 0,
            "missing_audio_rows": missing_audio_rows,
            "note": "No successful transcriptions were evaluated.",
        }
        with open(metrics_json_path, "w", encoding="utf-8") as handle:
            json.dump(summary, handle, indent=2)
        print(json.dumps(summary, indent=2))
        return pd.DataFrame(), summary

    results_df = pd.DataFrame(rows).sort_values("row_id").reset_index(drop=True)
    summary = {
        "samples_requested": int(len(df_meta)),
        "samples_with_audio": int(len(row_audio_map)),
        "samples_evaluated": int(len(results_df)),
        "missing_audio_rows": missing_audio_rows,
        "wer_mean": float(results_df["wer"].mean()),
        "cer_mean": float(results_df["cer"].mean()),
        "mer_mean": float(results_df["mer"].mean()),
        "wil_mean": float(results_df["wil"].mean()),
        "wip_mean": float(results_df["wip"].mean()),
        "rouge1_mean": float(results_df["rouge1"].mean()),
        "rouge2_mean": float(results_df["rouge2"].mean()),
        "rougeL_mean": float(results_df["rougeL"].mean()),
        "semantic_similarity_mean": float(results_df["semantic_similarity"].mean()),
        "wer_std": float(results_df["wer"].std(ddof=0)),
        "cer_std": float(results_df["cer"].std(ddof=0)),
    }

    results_df.to_csv(metrics_csv_path, index=False)
    with open(metrics_json_path, "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    print(f"Saved EKA transcription metrics CSV: {metrics_csv_path}")
    print(f"Saved EKA transcription summary JSON: {metrics_json_path}")
    print(json.dumps(summary, indent=2))

    return results_df, summary


# Run on all metadata rows that have a matching local clip in data/eka_dataset_audio
eka_results_df, eka_summary = evaluate_eka_transcription(max_samples=None, force_retranscribe=False)
display(eka_results_df.head(20))
eka_summary